In [0]:
from pyspark.sql.functions import col, current_timestamp

In [0]:
# Preâmbulo para configurar o catálogo
dbutils.widgets.text("catalog", "v_credit_case")
catalog = dbutils.widgets.get("catalog")

bronze_schema = f"{catalog}.bronze"

landing_path = f"/Volumes/{catalog}/bronze/landing"
checkpoint_path = f"/Volumes/{catalog}/bronze/checkpoints"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.`landing`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.`checkpoints`")

print("Catálogo configurado com sucesso!")

In [0]:
# Função para carregar um DataFrame a partir do nome de um arquivo
def load_df(file_pattern="*.csv", schema_key="default"):
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header","true")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schemas/{schema_key}")#
        .option("pathGlobFilter", file_pattern)#
        .load(landing_path)#
        .withColumn("source_file", col("_metadata.file_path"))
    )

In [0]:
# Função para salvar o DataFrame na camada bronze
def save_df(df, table_name):
    query = (
        df.withColumn("ingestion_datetime", current_timestamp()) 
            .writeStream
            .format("delta")
            .outputMode("append") 
            .option("checkpointLocation", f"{checkpoint_path}/{table_name}")
            .option("mergeSchema","true")
            .trigger(availableNow=True)
            .toTable(f"{bronze_schema}.{table_name}")
            )
    query.awaitTermination()
    print(f"Tabela {table_name} atualizada com sucesso!")